In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
import dash
from dash import dcc, html, Input, Output
import webbrowser

# =========================
# LOAD DATA
# =========================
base = Path("../data/gold")

df_country = pd.read_csv(base / "country_revenue.csv")
df_product = pd.read_csv(base / "product_revenue.csv")
df_monthly = pd.read_csv(base / "monthly_revenue.csv")
df_weekly = pd.read_csv(base / "weekly_revenue.csv")

df_monthly = df_monthly.drop(df_monthly.index[0], errors="ignore")

# =========================
# CLEAN COLUMNS
# =========================
for df in [df_country, df_product, df_monthly, df_weekly]:
    df.columns = df.columns.str.strip()

# =========================
# SAFE COLUMN DETECTION
# =========================
def find_col(df, keywords):
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

country_col_country = "Country"
country_col_revenue = find_col(df_country, ["revenue", "price", "total"])

prod_col_category = find_col(df_product, ["category", "product"])
prod_col_revenue = find_col(df_product, ["revenue", "price", "total"])

# =========================
# MONTHLY PREP
# =========================
df_monthly["Date"] = pd.to_datetime(
    df_monthly["Year"].astype(str) + "-" +
    df_monthly["Month"].astype(str) + "-01",
    errors="coerce"
)

df_monthly = df_monthly.dropna(subset=["Date"])
df_monthly = df_monthly.sort_values("Date")
df_monthly["MonthName"] = df_monthly["Date"].dt.strftime("%b")

# =========================
# WEEKLY PREP (NEW)
# =========================
if "Week" in df_weekly.columns:
    df_weekly["Week"] = df_weekly["Week"].astype(str)

# =========================
# NUMERIC CLEAN
# =========================
def safe_numeric(df, col):
    if col and col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

safe_numeric(df_country, country_col_revenue)
safe_numeric(df_product, prod_col_revenue)
safe_numeric(df_product, "TotalQuantity")
safe_numeric(df_monthly, "TotalPrice")
safe_numeric(df_weekly, "TotalPrice")

# =========================
# DASH APP
# =========================
app = dash.Dash(__name__)
server = app.server

# =========================
# OPTIONS
# =========================
country_options = [{"label": c, "value": c}
                   for c in df_country[country_col_country].dropna().unique()]

category_options = [{"label": c, "value": c}
                    for c in df_product[prod_col_category].dropna().unique()]

month_options = [{"label": m, "value": m}
                 for m in df_monthly["MonthName"].unique()]

# =========================
# EMPTY FIG
# =========================
def empty_fig(title):
    fig = px.scatter(title=title)
    fig.add_annotation(text="No data available", showarrow=False)
    return fig

# =========================
# LAYOUT
# =========================
app.layout = html.Div([

    html.H2("Revenue Dashboard", style={"textAlign": "center"}),

    # ================= FILTERS =================
    html.Div([

        dcc.Dropdown(id="country", options=country_options, multi=True, placeholder="Country"),
        dcc.Dropdown(id="category", options=category_options, multi=True, placeholder="Category"),

        dcc.Dropdown(
            id="time_type",
            options=[
                {"label": "Monthly", "value": "month"},
                {"label": "Weekly", "value": "week"}
            ],
            value="month",
            clearable=False
        ),

        dcc.Dropdown(
            id="top_n",
            options=[
                {"label": "Top 5", "value": 5},
                {"label": "Top 10", "value": 10},
                {"label": "Top 20", "value": 20},
            ],
            value=10,
            clearable=False
        ),

    ], style={
        "width": "80%",
        "margin": "auto",
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "gap": "10px"
    }),

    # ================= 2x2 GRID =================
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "gap": "15px",
        "padding": "20px"
    }, children=[

        html.Div([html.H5("Revenue by Country"),
                  dcc.Graph(id="map", style={"height": "420px"})]),

        html.Div([html.H5("Revenue by Category"),
                  dcc.Graph(id="product", style={"height": "420px"})]),

        html.Div([html.H5("Top Products"),
                  dcc.Graph(id="top_products", style={"height": "420px"})]),

        html.Div([html.H5("Revenue Trend"),
                  dcc.Graph(id="monthly", style={"height": "420px"})]),
    ])
])

# =========================
# CALLBACK
# =========================
@app.callback(
    Output("map", "figure"),
    Output("product", "figure"),
    Output("top_products", "figure"),
    Output("monthly", "figure"),
    Input("country", "value"),
    Input("category", "value"),
    Input("time_type", "value"),
    Input("top_n", "value"),
)
def update(countries, categories, time_type, top_n):

    dfc = df_country.copy()
    dfp = df_product.copy()

    if countries:
        dfc = dfc[dfc[country_col_country].isin(countries)]

    if categories:
        dfp = dfp[dfp[prod_col_category].isin(categories)]

    # ================= COUNTRY (OLD) =================
    df_map = dfc.groupby(country_col_country, as_index=False)[country_col_revenue].sum()
    fig_map = px.choropleth(df_map,
                            locations=country_col_country,
                            locationmode="country names",
                            color=country_col_revenue)

    # ================= CATEGORY (OLD) =================
    df_cat = dfp.groupby(prod_col_category, as_index=False)[prod_col_revenue].sum()
    fig_product = px.pie(df_cat, names=prod_col_category, values=prod_col_revenue, hole=0.3)

    # ================= TOP PRODUCTS (OLD) =================
    df_top = dfp.groupby(
        ["StockCode", "Description", prod_col_category],
        as_index=False
    ).agg({"TotalRevenue": "sum", "TotalQuantity": "sum"})

    df_top = df_top.sort_values("TotalRevenue", ascending=False).head(top_n)

    fig_top = px.bar(
        df_top,
        x="Description",
        y=["TotalRevenue", "TotalQuantity"],
        barmode="group"
    )

    # ================= TIME SWITCH (NEW) =================
    if time_type == "week":

        dfw = df_weekly.copy()

        fig_month = px.line(
            dfw.groupby("Week", as_index=False)["TotalPrice"].sum(),
            x="Week",
            y="TotalPrice",
            markers=True,
            title="Weekly Revenue Trend"
        )

    else:

        dfm = df_monthly.copy()

        dfm_grouped = dfm.groupby("MonthName", as_index=False)["TotalPrice"].sum()

        order = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"]

        dfm_grouped["MonthName"] = pd.Categorical(dfm_grouped["MonthName"], categories=order, ordered=True)
        dfm_grouped = dfm_grouped.sort_values("MonthName")

        fig_month = px.line(
            dfm_grouped,
            x="MonthName",
            y="TotalPrice",
            markers=True,
            title="Monthly Revenue Trend"
        )

    return fig_map, fig_product, fig_top, fig_month


# =========================
# RUN
# =========================
if __name__ == "__main__":
    webbrowser.open("http://127.0.0.1:8044")
    app.run(host="127.0.0.1", port=8044, debug=False)

: 